# part3.ipynb — Adversarial attacks

**Prerequisite:** Run `part1.ipynb` first.


### i220524 FAST-NUCES Assignment 2 — Responsible & Explainable AI


## Environment setup

Install dependencies (local or Colab), then continue. On Colab: **Runtime → Change runtime type → GPU**.



In [ ]:
"""Install dependencies via `pip install -r requirements.txt` in a fresh environment (e.g. Colab)."""


def ensure_pkg():
    """No-op placeholder; run pip manually or automate here if desired."""
    return None


ensure_pkg()


In [ ]:
"""Imports and global configuration for reproducibility.

Sets USE_TF=0 before HuggingFace loads, and forces trainer_utils.is_tf_available() to False so Trainer.setup never imports TensorFlow (broken TF wheels still report as installed).
"""
import os
os.environ["USE_TF"] = "0"
import re
import random
import zipfile
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.base import clone

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import transformers.trainer_utils as _hf_trainer_utils
_hf_trainer_utils.is_tf_available = lambda: False
from datasets import Dataset

from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ZIP_PATH = os.path.join(os.getcwd(), "jigsaw-unintended-bias-in-toxicity-classification.zip")
TRAIN_CSV_NAME = "train.csv"
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

BASELINE_DIR = os.path.join(ARTIFACT_DIR, "baseline_distilbert")
POISON_DIR = os.path.join(ARTIFACT_DIR, "poisoned_distilbert")
REWEIGHT_DIR = os.path.join(ARTIFACT_DIR, "reweighted_distilbert")
OVERSAMPLE_DIR = os.path.join(ARTIFACT_DIR, "oversample_distilbert")
ISOTONIC_PATH = os.path.join(ARTIFACT_DIR, "isotonic_calibrator.joblib")

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5

"""Optional: set True only for quick smoke tests (not for submission)."""
FAST_DEBUG = False
if FAST_DEBUG:
    NUM_EPOCHS = 1


## Load data from the zip (stratified 100k train / 20k eval)

We read only the columns needed to limit memory. **Label:** `y = (target >= 0.5).astype(int)`. `train_test_split` returns `(larger_slice, test_size_slice)`; the 20k holdout is the **second** return value.



In [ ]:
"""Load `train.csv` from the competition zip without extracting the full archive to disk."""


def read_train_from_zip(zip_path: str, csv_name: str) -> pd.DataFrame:
    """Read selected columns for memory efficiency and normalize names."""
    usecols = [
        "comment_text",
        "target",
        "black",
        "white",
        "muslim",
        "jewish",
        "homosexual_gay_or_lesbian",
    ]
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, usecols=usecols)
    df = df.rename(columns={"target": "toxic"})
    df["comment_text"] = df["comment_text"].fillna("").astype(str)
    for c in ["toxic", "black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df["y"] = (df["toxic"] >= 0.5).astype(int)
    return df


df_all = read_train_from_zip(ZIP_PATH, TRAIN_CSV_NAME)
if FAST_DEBUG:
    eval_n = 2000
    train_n = 5000
    cap = eval_n + train_n + 5000
    df_all, _ = train_test_split(
        df_all, train_size=min(cap, len(df_all)), stratify=df_all["y"], random_state=SEED
    )
else:
    eval_n = 20000
    train_n = 100000

y = df_all["y"].values
train_remaining, eval_df = train_test_split(
    df_all, test_size=eval_n, stratify=y, random_state=SEED
)
y_rem = train_remaining["y"].values
train_df, _discard = train_test_split(
    train_remaining, train_size=train_n, stratify=y_rem, random_state=SEED
)

print("train_df:", train_df.shape, "eval_df:", eval_df.shape)
print("toxic rate train:", train_df["y"].mean(), "eval:", eval_df["y"].mean())


In [ ]:
"""Reload baseline weights and rebuild tokenized datasets (run part1.ipynb first)."""
tokenizer = AutoTokenizer.from_pretrained(BASELINE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BASELINE_DIR)
model.eval()
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def tokenize_batch(examples, tokenizer_obj, max_length: int):
    """Tokenize comment batches for DistilBERT."""
    return tokenizer_obj(
        examples["comment_text"],
        truncation=True,
        max_length=max_length,
    )


train_ds = Dataset.from_pandas(train_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
eval_ds = Dataset.from_pandas(eval_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
train_ds = train_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)
eval_ds = eval_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)


def compute_metrics(eval_pred):
    """Compute accuracy, macro-F1, binary toxic F1, and AUC when defined."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    pr = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    out = {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "f1_binary": f1_score(labels, preds, pos_label=1),
    }
    try:
        out["auc"] = roc_auc_score(labels, pr)
    except ValueError:
        out["auc"] = float("nan")
    return out


args = TrainingArguments(
    output_dir=os.path.join(ARTIFACT_DIR, "trainer_reload"),
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=200,
    load_best_model_at_end=False,
    report_to=[],
    seed=SEED,
)


def hf_predict_proba_toxic(texts, m, tok, device_obj, max_len):
    """Return toxic-class probabilities for a list of strings."""
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device_obj)
    with torch.no_grad():
        logits = m(**enc).logits
        pr = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
    return pr


texts_eval = eval_df["comment_text"].tolist()
y_true = eval_df["y"].to_numpy()
scores = hf_predict_proba_toxic(texts_eval, model, tokenizer, device, MAX_LENGTH)
CHOSEN_THRESHOLD = 0.5
thr = CHOSEN_THRESHOLD
y_hat = (scores >= thr).astype(int)


# Part 3 — Adversarial attacks (from scratch)

**Attack 1:** `perturb(text)` implements ZWS insertion, Cyrillic homoglyphs, and random character duplication.

**Attack 2:** flip **5%** of training labels and retrain **from `distilbert-base-uncased`**, not from the Part 1 fine-tuned checkpoint.



In [ ]:
"""Character-level evasion attack evaluation (Attack 1)."""
HOMOGLYPH = str.maketrans(
    {
        "a": "а",
        "e": "е",
        "o": "о",
        "c": "с",
        "p": "р",
        "x": "х",
    }
)

TOXICISH = re.compile(
    r"\b(?:kill|die|hate|stupid|idiot|ugly|loser|moron|rape|nazi|terror)\w*\b",
    re.IGNORECASE,
)


def perturb(text: str) -> str:
    """Insert ZWS into toxic-looking tokens, apply homoglyphs, duplicate ~20% of chars."""
    if text is None:
        return ""

    def zws_word(w: str) -> str:
        """Insert zero-width spaces every 2–3 characters."""
        out = []
        step = 2
        i = 0
        while i < len(w):
            chunk = w[i : i + step]
            out.append(chunk)
            if i + step < len(w):
                out.append("​")
            i += step
            step = 3 if step == 2 else 2
        return "".join(out)

    def process_segment(seg: str) -> str:
        """Homoglyph map plus random intra-word duplication."""
        seg = seg.translate(HOMOGLYPH)
        out_chars: List[str] = []
        for ch in seg:
            out_chars.append(ch)
            if ch.isalpha() and random.random() < 0.20:
                out_chars.append(ch)
        return "".join(out_chars)

    pieces: List[str] = []
    last = 0
    for m in TOXICISH.finditer(text):
        pieces.append(process_segment(text[last : m.start()]))
        pieces.append(process_segment(zws_word(m.group(0))))
        last = m.end()
    pieces.append(process_segment(text[last:]))
    return "".join(pieces)


attack_pool = eval_df.copy()
attack_pool["score"] = scores
attack_pool["y_hat"] = y_hat
cand = attack_pool[(attack_pool["y_hat"] == 1) & (attack_pool["score"] >= 0.7)]
if len(cand) > 0:
    cand = cand.sample(n=min(500, len(cand)), random_state=SEED)
else:
    cand = cand

pert_texts = [perturb(t) for t in cand["comment_text"].tolist()]
scores_before = cand["score"].to_numpy() if len(cand) else np.array([])
scores_after = (
    hf_predict_proba_toxic(pert_texts, model, tokenizer, device, MAX_LENGTH) if len(cand) else np.array([])
)
y_after = (scores_after >= thr).astype(int) if len(cand) else np.array([])

asr = float(np.mean(y_after == 0)) if len(cand) > 0 else float("nan")
print("Attack-1 candidates:", len(cand))
print("ASR (fraction no longer flagged at same threshold):", asr)
print("mean confidence before:", float(np.mean(scores_before)) if len(cand) else float("nan"))
print("mean confidence after:", float(np.mean(scores_after)) if len(cand) else float("nan"))


In [ ]:
"""Label-flip poisoning retrain (Attack 2)."""
poison_df = train_df.copy()
rng = np.random.RandomState(SEED)
flip_mask = rng.rand(len(poison_df)) < 0.05
poison_df.loc[flip_mask, "y"] = 1 - poison_df.loc[flip_mask, "y"]

poison_ds = Dataset.from_pandas(poison_df[["comment_text", "y"]].rename(columns={"y": "labels"}))
poison_ds = poison_ds.map(
    lambda b: tokenize_batch(b, tokenizer, MAX_LENGTH),
    batched=True,
    remove_columns=["comment_text"],
)

model_poison = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_p = clone(args)
args_p.output_dir = os.path.join(ARTIFACT_DIR, "trainer_poison")

trainer_p = Trainer(
    model=model_poison,
    args=args_p,
    train_dataset=poison_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer_p.train()
trainer_p.save_model(POISON_DIR)

model_poison.eval()
model_poison.to(device)
scores_poison_eval = hf_predict_proba_toxic(texts_eval, model_poison, tokenizer, device, MAX_LENGTH)
y_poison_hat = (scores_poison_eval >= thr).astype(int)


def fnr_toxic(yt, yp):
    """False negative rate restricted to truly toxic comments."""
    tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
    return fn / (fn + tp) if (fn + tp) > 0 else float("nan")


compare = pd.DataFrame(
    [
        {
            "model": "clean",
            "accuracy": accuracy_score(y_true, y_hat),
            "f1_macro": f1_score(y_true, y_hat, average="macro"),
            "fnr_toxic": fnr_toxic(y_true, y_hat),
        },
        {
            "model": "poisoned",
            "accuracy": accuracy_score(y_true, y_poison_hat),
            "f1_macro": f1_score(y_true, y_poison_hat, average="macro"),
            "fnr_toxic": fnr_toxic(y_true, y_poison_hat),
        },
    ]
)
print(compare)


### Part 3 — Operational risk (model answer — tie to your ASR / metric table)

**Evasion** needs no access to training—only the ability to post text. It is **cheap to automate** (scripts, templates), scales with **volume**, and is the **default threat** for any public moderation API or open comment box. Impact is **per-comment** but can still be severe at scale (spam, coordinated campaigns).

**Poisoning** needs **write access to training data or the fine-tuning pipeline** (compromised MLOps, malicious insider, poisoned vendor dataset). It is **rarer** and harder to pull off, but can **shift model behavior for everyone** (systemic FNs/FPs), not just one post.

**Which is “more dangerous”?** For **typical external abuse**, **evasion** is the more **realistic and frequent** operational problem—defenses should include **input normalization**, **robustness testing**, and **human review** on edge cases. **Poisoning** is the more **realistic high-severity insider/supply-chain** scenario—defenses should emphasize **data provenance, access control, auditing, and retraining checks**. **Edit:** One sentence comparing **your ASR** vs **your poisoned-model FNR/F1 drop** to say which risk worries you more for *this* deployment.

